In [ ]:
# !pip install gradio openai pypdf openai-agents

In [5]:
!pip install python-dotenv

In [43]:
import json
import os

import requests
from agents import Agent, AsyncOpenAI, ModelSettings, OpenAIChatCompletionsModel, RunConfig, Runner, function_tool, set_tracing_disabled
from IPython.display import Markdown, display
from pypdf import PdfReader

In [44]:
import os
from dotenv import load_dotenv

load_dotenv()

MOONSHOT_API_KEY = os.getenv("MOONSHOT_API_KEY")
OLOSTEP_API_KEY = os.getenv("OLOSTEP_API_KEY")

print("Kimi key loaded:", bool(MOONSHOT_API_KEY))
print("Olostep key loaded:", bool(OLOSTEP_API_KEY))

Kimi key loaded: True
Olostep key loaded: True


In [45]:
KIMI_MODEL = "kimi-k2.6"
KIMI_BASE_URL = "https://api.moonshot.ai/v1"
OLOSTEP_SEARCH_URL = "https://api.olostep.com/v1/searches"
OLOSTEP_SCRAPE_URL = "https://api.olostep.com/v1/scrapes"
MAX_AGENT_TURNS = 25


cv_path = "Rif-resume.pdf"
preferences = """ Remote digital marketing roles, including Digital Marketing Specialist, Performance Marketing Specialist, 
Social Media Marketing Executive, E-commerce Marketing Specialist, Creative/Content Marketing Specialist, 
Marketing Executive, Growth Marketing Associate, Brand Marketing Specialist, 
Ad Operations Specialist, and Marketing Coordinator. """.strip()

set_tracing_disabled(True)

In [46]:
kimi_client = AsyncOpenAI(
   api_key=os.environ["MOONSHOT_API_KEY"],
   base_url=KIMI_BASE_URL,
)

In [47]:
kimi_model = OpenAIChatCompletionsModel(
    model = KIMI_MODEL,
    openai_client = kimi_client,
)

In [48]:
model_settings = ModelSettings(
    tool_choice = "auto",
    parallel_tool_calls = True,
    extra_body = {"thinking": {"type" : "disabled"}},
)

In [49]:
run_config = RunConfig(
    workflow_name = "JobFinder AI KIMI Search",
    tracing_disabled = True
)

In [50]:
reader = PdfReader(cv_path)
cv_text = "\n".join(page.extract_text() or "" for page in reader.pages)[:12000]
print(f"Loaded {len(cv_text):,} characters from {cv_path}")

Loaded 2,273 characters from Rif-resume.pdf


## Agent Instruction

In [51]:
AGENT_INSTRUCTIONS = """
You are JobFinder AI, a focused job-search agent.

Tool plan:
- Call search_jobs exactly once with limit 8.
- Read at most 3 direct job pages with read_job_page.
- After reading up to 3 pages, stop using tools and write the report.
- Search again only if the first search returns zero usable jobs.
- Avoid broad search pages, expired jobs, and LinkedIn unless no better source exists.

Report rules:
- Keep the report simple, clear, and practical.
- Use short bullets.
- Do not use em dashes.
- Do not use contractions.
- Do not add text before or after the report.
- End after the final Job Notes entry.
- Include at least 5 ranked jobs if the search results contain at least 5 usable jobs.
- If only 3 pages were scraped, use backup jobs from search results when they look usable.
- Every job must include a clickable Markdown link.
- Every job must have one apply decision: Apply, Maybe, or Do not apply.

Use exactly this Markdown structure:

# JobFinder AI Report

## Best Match

- **Role:** <job title>
- **Company:** <company>
- **Apply decision:** Apply / Maybe / Do not apply
- **Fit score:** <score>/100
- **Link:** [Apply here](<job url>)

**Why this is the best match:**

- <specific reason>
- <specific reason>
- <specific reason>

## Ranked Jobs

| Rank | Role | Company | Apply? | Fit | Link |
| --- | --- | --- | --- | --- | --- |
| 1 | <role> | <company> | Apply / Maybe / Do not apply | <score>/100 | [Apply here](<url>) |

## Job Notes

### 1. <Role> at <Company>

- **Apply decision:** Apply / Maybe / Do not apply
- **Fit score:** <score>/100
- **Link:** [Apply here](<job url>)

**Why it fits:**

- <bullet>
- <bullet>

**Concerns:**

- <bullet>
- <bullet>

**Application angle:**

- <how the person should position their CV/application>
""".strip()

## Runtime Prompt

In [52]:
RUN_PROMPT_TEMPLATE = """
Find current job postings for this candidate and rank them by fit.

Keep the run simple:
- one search
- up to three page reads
- final report

The final report must follow AGENT_INSTRUCTIONS exactly.
Use simple wording. Do not use em dashes. Do not use contractions.

Candidate CV:
{cv_text}

Preferences:
{preferences}
""".strip()

## Tool to search the web

In [53]:
@function_tool
def search_jobs(query: str, limit: int = 8) -> str:
    """Search the web for job listings and return compact JSON results."""
    response = requests.post(
        OLOSTEP_SEARCH_URL,
        headers={"Authorization": f"Bearer {os.environ['OLOSTEP_API_KEY']}", "Content-Type": "application/json"},
        json={"query": query},
        timeout=60,
    )
    response.raise_for_status()
    links = response.json().get("result", {}).get("links", [])[:limit]
    results = [
        {"title": item.get("title", "Untitled"), "url": item.get("url"), "description": item.get("description", "")}
        for item in links
        if isinstance(item, dict) and item.get("url")
    ]
    return json.dumps(results, ensure_ascii=False)

## Tool to read the job page

In [54]:
@function_tool
def read_job_page(url: str) -> str:
    """Scrape one job listing URL and return markdown text."""
    response = requests.post(
        OLOSTEP_SCRAPE_URL,
        headers={"Authorization": f"Bearer {os.environ['OLOSTEP_API_KEY']}", "Content-Type": "application/json"},
        json={"url_to_scrape": url, "formats": ["markdown"]},
        timeout=120,
    )
    response.raise_for_status()
    markdown = response.json().get("result", {}).get("markdown_content") or ""
    return markdown[:8000]

## Create a JobFinder AI  Agent

In [55]:
agent = Agent(
    name = "JobFinder AI",
    model = kimi_model,
    model_settings = model_settings,
    tools = [search_jobs, read_job_page],
    instructions = AGENT_INSTRUCTIONS,
)

In [56]:
prompt = RUN_PROMPT_TEMPLATE.format(cv_text=cv_text, preferences=preferences)

In [57]:
print("Starting agent run")


result = Runner.run_streamed(
   agent,
   prompt,
   max_turns=MAX_AGENT_TURNS,
   run_config=run_config,
)

Starting agent run


## Streaming Loop

In [61]:
async for event in result.stream_events():
   if event.type == "agent_updated_stream_event":
       print(f"Agent: {event.new_agent.name}")
   elif event.type == "run_item_stream_event":
       item = event.item
       if event.name == "tool_called":
           raw = item.raw_item
           tool_name = raw.get("name") if isinstance(raw, dict) else getattr(raw, "name", "tool")
           arguments = raw.get("arguments") if isinstance(raw, dict) else getattr(raw, "arguments", "")
           arguments = str(arguments).replace(chr(10), " ")[:500]
           print(f"Tool call: {tool_name}")
           if arguments:
               print(f"Parameters: {arguments}")
       elif event.name == "tool_output":
           print(f"Tool output: {len(str(item.output)):,} chars")
       elif event.name == "message_output_created":
           print("Final message ready")

RateLimitError: Error code: 429 - {'error': {'message': 'Your account org-88f4758b3ba447a89fe333acdd0e536b<ak-fcdfemynf5b111hebw91> request reached organization max RPM: 3, please try again after 1 seconds', 'type': 'rate_limit_reached_error'}}

In [62]:
report = result.final_output
globals()["report"] = report


print("Run complete")
print(f"Model responses: {len(result.raw_responses)}")
print(f"Run items: {len(result.new_items)}")
print(f"Final output: {len(str(report)):,} chars")

Run complete
Model responses: 3
Run items: 6
Final output: 4 chars


In [63]:
display(Markdown(report))

<IPython.core.display.Markdown object>